# Import Libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# EDA
 

In [ ]:
fd = pd.read_csv('creditcard.csv\\creditcard.csv')
print("\n statistical summary of key features:")
print(fd[['Time', 'Amount', 'V1', 'V2', 'Class']].describe().round(2))
fd.shape

In [ ]:
fd.head()


In [ ]:
fd.tail()

In [ ]:
print(fd.isnull().sum())

In [ ]:

fd.info()

In [ ]:
fd.duplicated().any()
fd = fd.drop_duplicates()

In [ ]:
# Correlation BEFORE feature engineering
base_features = [col for col in fd.columns if col.startswith('V')] + ['Amount', 'Time', 'Class']
corr_before = fd[base_features].corr()['Class'].drop('Class').sort_values()
plt.figure(figsize=(10, 6))
corr_before.plot(kind='barh', color='steelblue')
plt.title('Correlation with Fraud (Before Feature Engineering)')
plt.xlabel('Correlation')
plt.tight_layout()
plt.show()


# Insights correlation features
1. V17, V14, V12, and V10 show strong negative correlations, meaning lower values in these features are highly associated with fraudulent transactions.
2. V4 and V11 show positive correlation, the model to flag potential fraud.
3. Amount with Class is near zero,suggesting that "Amount" alone is not a strong linear predictor of fraud.
 4. Time shows a small negative correlation with Class,so it do not provide a linear predictive signal for fraud.

In [ ]:
# Correlation HEATMAP (V features only)
plt.figure(figsize=(8, 6))
v_features = [col for col in fd.columns if col.startswith('V')][:15]  # Top 15 V features
corr_matrix = fd[v_features].corr()
sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', center=0, 
            square=True, cbar_kws={'label': 'Correlation'})
plt.title('V Features Correlation Heatmap (PCA Components)')
plt.tight_layout()
plt.show()


# PCA confirmation
The heatmap shows near-zero correlation (neutral color) between all pairs of V features

The only strong correlation (1.0) exists on the diagonal, representing each feature's correlation with itself

In [ ]:
fd.shape
fd['Class'].value_counts()

In [ ]:
# General statistics for Amount
print("General Amount Statistics:")
print(fd['Amount'].describe().round(2))

print("\nAmount Statistics by Class:")
print(fd.groupby('Class')['Amount'].describe().round(2))

fig = plt.figure(figsize=(16, 10))
#distribution of Amount
ax1 = plt.subplot(2, 2, 1)
sns.histplot(fd['Amount'], bins=100, kde=True, ax=ax1, color='skyblue')
ax1.set_title('Overall Amount Distribution (Frequency)')
ax1.set_xlabel('Amount ($)')

# Boxplot for  amount by class
ax2 = plt.subplot(2, 2, 2)
sns.boxplot(x='Class', y='Amount', data=fd, ax=ax2)
ax2.set_ylim(0, 500) 
ax2.set_title('Amount Distribution: Normal (0) vs Fraud (1)')

# Density Plot for Amount by class

ax3 = plt.subplot(2, 1, 2)
fd[fd['Class']==0]['Amount'].plot.density(ax=ax3, label='Non_Fraud', color='#2ecc71', lw=2)
fd[fd['Class']==1]['Amount'].plot.density(ax=ax3, label='Fraud', color="#e74c3c", lw=2)
ax3.set_title('Density Estimation: Fraud vs Non-Fraud Amount')
ax3.set_xlabel('Amount ($)')
ax3.set_xlim(0, 500)
ax3.legend()

plt.tight_layout()
plt.show()

1. The median fraud amount ($9.82) is lower than the legitimate median ($22.00),fraudsters execute small transactions to verify card validity without triggering high-value alerts.

2. Both fraud and normal transactions have a very high Standard Deviation (~250),This means the amounts are widely spread and overlap

In [ ]:
#plot class distribution
sns.countplot(x='Class', data=fd)
plt.title('Distribution of Class variable')
plt.show()


In [ ]:
plt.pie(fd['Class'].value_counts(), labels=['non_fraudulent', 'Fraudulent'], autopct='%1.2f%%', colors=['lightblue', 'salmon'])
plt.title('Proportion of non_fraudulent vs Fraudulent Transactions')
plt.show()


# Featur Engineering
Engineer Time (convert raw time into meaningful info)

In [ ]:
## Convert raw seconds into a 24-hour cycle feature
fd['Hour'] = (fd['Time'] // 3600) % 24
# Transform 'Hour' into cyclical coordinates (Sin and Cos) 
fd['hour_sin'] = np.sin(2 * np.pi * fd['Hour'] / 24)
fd['hour_cos'] = np.cos(2 * np.pi * fd['Hour'] / 24)
fd.drop('Time', axis=1, inplace=True)

# Preprocessing

In [ ]:
from sklearn.model_selection import StratifiedKFold
# 3. K-Fold Target Encoding for Hourly Fraud Risk
fd['hourly_fraud_risk'] = np.nan
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for train_idx, val_idx in skf.split(fd, fd['Class']):
    # Use the training folds to calculate the mean fraud rate per hour
    train_fold = fd.iloc[train_idx]
    hourly_means = train_fold.groupby('Hour')['Class'].mean()
    # Map means onto the validation fold
    fd.loc[fd.index[val_idx], 'hourly_fraud_risk'] = fd.loc[fd.index[val_idx], 'Hour'].map(hourly_means)
# Fill any NaN values (hours that might not have appeared in a specific fold with 0.17)
fd['hourly_fraud_risk'] = fd['hourly_fraud_risk'].fillna(fd['Class'].mean())
print("New Time Features Created: hour_sin, hour_cos, hourly_fraud_risk")

In [ ]:
# Hourly Fraud Analysis
fd['Hour'] = fd['Hour'].astype(int)
hourly_counts = fd.groupby(['Hour', 'Class']).size().reset_index(name='Count')
fraud_rate = fd.groupby('Hour')['Class'].mean() * 100
avg_rate = fraud_rate.mean()
hours = np.arange(24)
fig, axes = plt.subplots(3, 1, figsize=(8, 6), sharex=True)

# Plot 1: Frequency 
sns.barplot(
    data=hourly_counts,
    x='Hour',
    y='Count',
    hue='Class',
    order=hours,
    palette={0: 'steelblue', 1: 'darkred'},
    ax=axes[0]
)
axes[0].set_title('Transaction Frequency by Hour', fontsize=13)
axes[0].set_ylabel('Transactions')
axes[0].legend(title='Class', labels=['Normal', 'Fraud'])

# Plot 2: Fraud Rate 
sns.barplot(
    x=fraud_rate.index,
    y=fraud_rate.values,
    order=hours,
    color='darkred',
    ax=axes[1]
)
axes[1].axhline(avg_rate, linestyle='--', color='black', label='Average')
axes[1].set_title('Fraud Rate per Hour', fontsize=13)
axes[1].set_ylabel('Fraud Rate (%)')
axes[1].legend()

#  Plot 3: Fraud Risk 
sns.lineplot(
    x=fraud_rate.index,
    y=fraud_rate.values,
    marker='o',
    linewidth=2,
    color='darkred',
    ax=axes[2]
)

axes[2].axhline(avg_rate, linestyle='--', color='black', label='Average')

for hour, rate in fraud_rate.items():
    if rate > avg_rate * 1.5:
        axes[2].scatter(hour, rate, color='black', s=70, zorder=3)
        axes[2].text(
            hour,
            rate + avg_rate * 0.05,
            f'{rate:.2f}%',
            ha='center',
            va='bottom',
            fontsize=9,
            fontweight='bold',
            bbox=dict(facecolor='white', alpha=0.8, edgecolor='none')
        )

axes[2].set_title('High-Risk Hours', fontsize=13)
axes[2].set_xlabel('Hour of Day')
axes[2].set_ylabel('Fraud Rate (%)')
axes[2].legend()


axes[2].set_xlim(-0.5, 23.5)
axes[2].set_xticks(hours)
axes[2].set_xticklabels(hours)

plt.tight_layout()
plt.show()


# Insights
 1. Transaction Frequency
- Most transactions occur during **business hours** (9 AM – 6 PM). 
 2. Actual Fraud Rate
- The **highest fraud risk** occurs at specific hours:  
  - **2 AM – 4 AM:** Fraud rate spikes above 1.5× the daily average → likely targeting low-activity periods.  
  - **10 PM – 11 PM:** Slight increase in risk
 3. volume
  High transaction hours are not always the riskiest.
- Daytime hours (9 AM – 6 PM) have the most transactions but **lower fraud probability**.  
- **Nighttime monitoring** ( 2–4 AM) is crucial for fraud  



In [ ]:
features_v = [col for col in fd if col.startswith("V")]
print("PCA features:", features_v)
feature_importance = []
for feat in features_v:
    fraud_data = fd[fd['Class'] == 1][feat]
    normal_data = fd[fd['Class'] == 0][feat]
#
    diff = np.abs(fraud_data.mean() - normal_data.mean())
    pooled_std = np.sqrt((fraud_data.std()**2 + normal_data.std()**2) / 2)
    effect_size = diff / pooled_std
    feature_importance.append({'Feature': feat, 'Effect_Size': effect_size})
importance_df = pd.DataFrame(feature_importance).sort_values(by='Effect_Size', ascending=False)
top_5_features = importance_df.head(5)

print("Top 5 Most Discriminative Features:")
print(top_5_features)
plt.figure(figsize=(20, 5 * len(top_5_features)))
for i, row in top_5_features.iterrows():
    feat = row['Feature']
    d = row['Effect_Size']

    # Plot 1: Boxplot
    ax1 = plt.subplot(len(top_5_features), 2, 2 * list(top_5_features.index).index(i) + 1)
    sns.boxplot(
    x='Class',
    y=feat,
    hue='Class',
    data=fd,
    palette=['skyblue', 'red'],
    legend=False,
    ax=ax1
)

    ax1.set_title(f'{feat} Distribution by Class (Cohen\'s d = {d:.2f})')
    ax1.set_xlabel('Class (0 = Normal, 1 = Fraud)')

    # Plot 2: Mean trend over time
    ax2 = plt.subplot(len(top_5_features), 2, 2 * list(top_5_features.index).index(i) + 2)
    hourly_mean = fd.groupby(['Hour', 'Class'])[feat].mean().reset_index()
    sns.lineplot(
        data=hourly_mean,
        x='Hour',
        y=feat,
        hue='Class',
        marker='o',
        palette=['skyblue', 'red'],
        ax=ax2
    )
    ax2.set_title(f'{feat} Mean Value vs Hour')
    ax2.set_xlabel('Hour')

plt.tight_layout()
plt.show()


In [ ]:
top_features = ['V14', 'V4', 'V12', 'V11', 'V10']
means_df = fd.groupby('Class')[top_features].mean().T
means_df.columns = ['Normal_Mean (Class 0)', 'Fraud_Mean (Class 1)']
means_df['Absolute_Difference'] = abs(means_df['Normal_Mean (Class 0)'] - means_df['Fraud_Mean (Class 1)'])
print("Numerical Comparison of Top Features:")
print(means_df.round(3))

Statistical Separation:

 Features like V14, V4, and V12 show a massive "Effect Size" (Cohen's d > 1.8), meaning they are the strongest indicators for distinguishing fraud from normal activity.

 Positive Indicators (V4, V11): Fraudulent transactions usually have higher values than normal ones.

Negative Indicators (V14, V12, V10): Fraudulent transactions usually have significantly lower (negative) values.

Behavioral Trends (Right Plots):  The Blue Line (Normal) is always stable near zero, showing consistent human behavior.

The Red Line (Fraud) deviates heavily, especially during the "High-Risk" hours identified earlier (e.g., 2 AM – 5 AM).

In [ ]:
# feature-target split
X = fd.drop('Class', axis=1)
Y = fd['Class']
print("X.shape" , X.shape)
print("Y Distribution:\n", Y.value_counts())

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, stratify=Y, random_state=42)
print("X_train.shape:", X_train.shape)
print("X_test.shape:", X_test.shape)

In [ ]:
from sklearn.preprocessing import RobustScaler
scaler = RobustScaler() 
X_train['Amount_scaled'] = scaler.fit_transform(X_train[['Amount']])
X_test['Amount_scaled'] = scaler.transform(X_test[['Amount']])
# drop original Amount
X_train = X_train.drop('Amount', axis=1)
X_test = X_test.drop('Amount', axis=1)

In [ ]:
X_train = X_train.drop('Hour', axis=1)
X_test = X_test.drop('Hour', axis=1)
X_train.columns


# Baseline models (without SMOTE)

In [ ]:
from sklearn.linear_model import LogisticRegression 
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score)

# Define models
models = {
    "Logistic Regression": LogisticRegression(max_iter=100, class_weight='balanced', random_state=42),
    "Decision Tree": DecisionTreeClassifier(class_weight='balanced', random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
}


# Training and evaluating models without SMOTE
for model_name, model in models.items():
    print(f"\n====Training {model_name}====(NO SMOTE)")
    model.fit(X_train, Y_train)
    Y_pred = model.predict(X_test)
    Y_proba = model.predict_proba(X_test)[:, 1]
    print("Confusion Matrix:\n", confusion_matrix(Y_test, Y_pred))
    print("Classification Report:\n", classification_report(Y_test, Y_pred))
    print("ROC-AUC Score:", roc_auc_score(Y_test, Y_proba))

# Advanced model XGBOOST without SMOTE

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score)
# Initializing XGBoost Classifier with class weight adjustment
xgb = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight= Y_train.value_counts()[0] / Y_train.value_counts()[1],
    eval_metric='logloss',
    early_stopping_rounds=20,
    random_state=42
)

# Training XGBoost Classifier 
xgb.fit(X_train, Y_train, eval_set=[(X_test, Y_test)], verbose=False)
Y_pred = xgb.predict(X_test)
Y_proba = xgb.predict_proba(X_test)[:, 1]
# Evaluation metrics

print("====XGBoost Classifier==== (NO SMOTE)")
print("Confusion Matrix:\n", confusion_matrix(Y_test, Y_pred))
print("Classification Report:\n", classification_report(Y_test, Y_pred))
print("ROC-AUC Score:", roc_auc_score(Y_test, Y_proba))


In [ ]:
results = xgb.evals_result()
plt.plot(results['validation_0']['logloss'], label='Validation LogLoss')
plt.axvline(xgb.best_iteration, color='red', linestyle='--', label='Best Iteration')
plt.xlabel('Boosting Round')
plt.ylabel('LogLoss')
plt.legend()
plt.show()


In [ ]:
# Feature Importance from XGBoost
feature_names = X_train.columns
importances = xgb.feature_importances_
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False)

top_15 = importance_df.head(15)

print("Top 15 Most Important Features:")
print(top_15)
plt.figure(figsize=(10, 6))
plt.barh(top_15['Feature'], top_15['Importance'], color='steelblue')
plt.xlabel('Importance Score')
plt.title('Top 15 Most Important Features - XGBoost (NO SMOTE)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from xgboost import XGBClassifier
print("=== 5-Fold Cross-Validation (XGBoost NO SMOTE) ===")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# Create a new XGBClassifier without early_stopping_rounds for CV
xgb_cv = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight= Y_train.value_counts()[0] / Y_train.value_counts()[1],
    eval_metric='logloss',
    random_state=42
)

cv_scores = cross_val_score(xgb_cv, X_train, Y_train, 
                            cv=skf, scoring='roc_auc')
print(f"CV Scores: {cv_scores}, Mean: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")


In [ ]:
cm_xgb = confusion_matrix(Y_test, xgb.predict(X_test))
sns.heatmap(cm_xgb, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix for XGBoost (NO SMOTE)')
plt.show()

# Handling Class Imbalance SMOTE for Only Train

In [ ]:
# Applying SMOTE to the training data
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
X_train_smote, Y_train_smote = smote.fit_resample(X_train, Y_train)
print("Before SMOTE:", Y_train.value_counts())
print("After SMOTE:", Y_train_smote.value_counts())

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12,4))
sns.countplot(x=Y_train, ax=ax1)
ax1.set_title('Before SMOTE (Training)')
sns.countplot(x=Y_train_smote, ax=ax2)
ax2.set_title('After SMOTE (Training)')
plt.tight_layout()
plt.show()

# Baseline models (With SMOTE)

In [ ]:
# Training and evaluating models with SMOTE
models_smote = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
}

for model_name, model in models_smote.items():
    model.fit(X_train_smote, Y_train_smote)
    Y_pred = model.predict(X_test)
    Y_proba = model.predict_proba(X_test)[:, 1]

    print(f"\n====Training {model_name}==== (WITH SMOTE)")
    print("Confusion Matrix:\n", confusion_matrix(Y_test, Y_pred))
    print("Classification Report:\n", classification_report(Y_test, Y_pred))
    print("ROC-AUC Score:", roc_auc_score(Y_test, Y_proba))

In [ ]:
cm_rf_smote = confusion_matrix(Y_test, models_smote["Random Forest"].predict(X_test))
sns.heatmap(cm_rf_smote, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix for Random Forest (WITH SMOTE)')
plt.show()

In [ ]:
importances = models_smote["Random Forest"].feature_importances_
feature_names = X_train_smote.columns 
feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance Score': importances})
feature_importance_df = feature_importance_df.sort_values(by='Importance Score', ascending=False).head(15)
plt.figure(figsize=(10, 6))
sns.barplot(x='Importance Score', y='Feature', data=feature_importance_df, color='steelblue')
plt.title('Top 15 Important Features - Random Forest')
plt.show()

# Advanced model XGBOOST With SMOTE

In [ ]:
xgb_smote = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    early_stopping_rounds=20,
    random_state=42
)
xgb_smote.fit(X_train_smote, Y_train_smote , eval_set=[(X_test, Y_test)], verbose=False)
Y_pred = xgb_smote.predict(X_test)
Y_proba = xgb_smote.predict_proba(X_test)[:, 1]
print("====XGBoost Classifier==== (WITH SMOTE)")
print("Confusion Matrix:\n", confusion_matrix(Y_test, Y_pred))
print("Classification Report:\n", classification_report(Y_test, Y_pred))
print("ROC-AUC Score:", roc_auc_score(Y_test, Y_proba))

In [ ]:
from sklearn.metrics import roc_curve
models_to_compare = ['RANDOM FOREST WITH SMOTE', 'XGBoost WITHOUT SMOTE']
probas = [models_smote["Random Forest"].predict_proba(X_test)[:,1], 
          xgb.predict_proba(X_test)[:,1]]

plt.figure(figsize=(8,6))
for i, name in enumerate(models_to_compare):
    fpr, tpr, _ = roc_curve(Y_test, probas[i])
    plt.plot(fpr, tpr, label=f'{name} (AUC={roc_auc_score(Y_test, probas[i]):.3f})')
plt.plot([0,1],[0,1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison (RANDOM FOREST WITH SMOTE VS XGBOOST WITHOUT SMOTE)')
plt.legend()
plt.show()

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score
plt.figure(figsize=(8,6))
for i, name in enumerate(models_to_compare):
    precision, recall, _ = precision_recall_curve(Y_test, probas[i])
    avg_precision = average_precision_score(Y_test, probas[i])
    plt.plot(recall, precision, label=f'{name} (PR-AUC={avg_precision:.3f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve (RANDOM FOREST WITH SMOTE VS XGBOOST WITHOUT SMOTE)')
plt.legend()
plt.show()

In [ ]:
#Find BEST threshold for XGBoost NO SMOTE & Random Forest
from sklearn.metrics import f1_score
y_proba_xgb = xgb.predict_proba(X_test)[:, 1]  # XGBoost
y_proba_rf = models_smote["Random Forest"].predict_proba(X_test)[:, 1]  # RF
thresholds = np.arange(0.1, 0.9, 0.01)

# For XGBoost
f1_scores_xgb = []
for threshold in thresholds:
    y_pred = (y_proba_xgb >= threshold).astype(int)
    f1_scores_xgb.append(f1_score(Y_test, y_pred))
best_idx_xgb = np.argmax(f1_scores_xgb)
best_threshold_xgb = thresholds[best_idx_xgb]
best_f1_xgb = f1_scores_xgb[best_idx_xgb]

print("=== XGBoost NO SMOTE ===")
print(f"Default (0.5): F1={f1_score(Y_test, (y_proba_xgb >= 0.5).astype(int)):.3f}")
print(f"BEST: {best_threshold_xgb:.3f} → F1={best_f1_xgb:.3f}")

# For Random Forest
f1_scores_rf = []
for threshold in thresholds:
    y_proba = y_proba_rf  # Use RF proba
    y_pred = (y_proba >= threshold).astype(int)
    f1_scores_rf.append(f1_score(Y_test, y_pred))
best_idx_rf = np.argmax(f1_scores_rf)
best_threshold_rf = thresholds[best_idx_rf]
best_f1_rf = f1_scores_rf[best_idx_rf]

print("\n=== Random Forest ===")
print(f"Default (0.5): F1={f1_score(Y_test, (y_proba_rf >= 0.5).astype(int)):.3f}")
print(f"BEST: {best_threshold_rf:.3f} → F1={best_f1_rf:.3f}")

# Plot 1: XGBoost
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(thresholds, f1_scores_xgb, 'b-', linewidth=2)
plt.axvline(best_threshold_xgb, color='red', linestyle='--', 
           label=f'Best: {best_threshold_xgb:.3f}')
plt.xlabel('Threshold'); plt.ylabel('F1'); plt.title('XGBoost NO SMOTE')
plt.legend(); plt.grid(True, alpha=0.3)

# Plot 2: Random Forest
plt.subplot(1, 2, 2)
plt.plot(thresholds, f1_scores_rf, 'g-', linewidth=2)
plt.axvline(best_threshold_rf, color='red', linestyle='--', 
           label=f'Best: {best_threshold_rf:.3f}')
plt.xlabel('Threshold'); plt.ylabel('F1'); plt.title('Random Forest')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
import shap
explainer = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_test)
# Summary plot to show the impact of each feature
shap.summary_plot(shap_values, X_test)

In [ ]:
fraud_sample = pd.concat([
    fd[fd['Class'] == 1].sample(50, random_state=42),
    fd[fd['Class'] == 0].sample(950, random_state=42)
]).sample(frac=1, random_state=42).reset_index(drop=True)

fraud_sample['Transaction_ID'] = fraud_sample.index + 1

fraud_sample.to_csv('fraud_small_sample.csv', index=False)

print("Shuffled fraud sample created")


In [ ]:
import joblib
joblib.dump(xgb, 'XGboost_model_without_smote.pkl')
joblib.dump(best_threshold_xgb, 'XGboost_best_threshold.pkl')
joblib.dump(models_smote["Random Forest"], 'random_forest_final.pkl')
joblib.dump(best_threshold_rf, 'rf_threshold.pkl')
joblib.dump(scaler, 'scaler.pkl')
hourly_fraud_risk_map = fd.groupby('Hour')['Class'].mean()
hourly_fraud_risk_map.to_csv('hourly_fraud_risk_map.csv')